# Dementia Assessment Training V2 - IMPROVED

**🚀 This version includes advanced techniques to fix the class prediction problem!**

## What's New in V2

✅ **Class Weighting**: 2x penalty for misclassifying dementia
✅ **Oversampling**: Balanced 50/50 training set
✅ **Better Metrics**: F1-macro + per-class recall
✅ **Data Augmentation**: Noise, time stretch, pitch shift
✅ **Longer Audio**: 15 seconds instead of 10
✅ **More Epochs**: 15 instead of 10

## Previous Problem

The original model only predicted "nodementia" (77% accuracy but 0% recall for dementia).
This version forces the model to learn BOTH classes.

**See [TRAINING_ANALYSIS.md](TRAINING_ANALYSIS.md) for detailed problem analysis**

In [23]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"  # Use GPU 6 only

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
from datasets import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import json
import audiomentations as A
from datetime import datetime

/mnt/bst/a100/mkarakay/aozbulut/venvs/dementia_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## CONFIGURATION

In [4]:
USE_COMBINED_DATASET = True
SAMPLING_RATE = 16000
MAX_DURATION = 15.0  # Increased from 10s
MODEL_CHECKPOINT = "facebook/wav2vec2-base"
EPOCHS = 15  # Increased from 10
BATCH_SIZE = 8
LEARNING_RATE = 2e-5  # Reduced for stability

# Advanced techniques
USE_CLASS_WEIGHTING = True
USE_OVERSAMPLING = True
USE_AUGMENTATION = True

# Paths
DATA_DIR = "./data"
if USE_COMBINED_DATASET:
    TRAIN_CSV = os.path.join(DATA_DIR, "train_dm_combined.csv")
    VALID_CSV = os.path.join(DATA_DIR, "valid_dm_combined.csv")
    TEST_CSV = os.path.join(DATA_DIR, "test_dm_combined.csv")
    OUTPUT_DIR = "./results/wav2vec2-base-improved-v2"
else:
    TRAIN_CSV = os.path.join(DATA_DIR, "train_dm_new.csv")
    VALID_CSV = os.path.join(DATA_DIR, "valid_dm_new.csv")
    TEST_CSV = os.path.join(DATA_DIR, "test_dm_new.csv")
    OUTPUT_DIR = "./results/wav2vec2-base-improved-dementianet-v2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## SETUP LOGGING

In [5]:

print("="*70)
print("DEMENTIA ASSESSMENT TRAINING V2 - IMPROVED")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nConfiguration:")
print(f"  Model: {MODEL_CHECKPOINT}")
print(f"  Audio: {MAX_DURATION}s @ {SAMPLING_RATE} Hz")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"\nEnhancements:")
print(f"  Class weighting: {USE_CLASS_WEIGHTING}")
print(f"  Oversampling: {USE_OVERSAMPLING}")
print(f"  Augmentation: {USE_AUGMENTATION}")
print(f"\nOutput: {OUTPUT_DIR}")
print("="*70)

# Check GPU
print(f"\nGPU Info:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

DEMENTIA ASSESSMENT TRAINING V2 - IMPROVED
Start time: 2026-02-25 17:26:56

Configuration:
  Model: facebook/wav2vec2-base
  Audio: 15.0s @ 16000 Hz
  Epochs: 15
  Batch size: 8
  Learning rate: 2e-05

Enhancements:
  Class weighting: True
  Oversampling: True
  Augmentation: True

Output: ./results/wav2vec2-base-improved-v2

GPU Info:
  PyTorch: 2.6.0+cu124
  CUDA available: True
  GPU: NVIDIA A100-SXM4-80GB
  Memory: 85.09 GB


## LOAD DATA

In [6]:
print("\n" + "="*70)
print("LOADING DATA")
print("="*70)

train_df = pd.read_csv(TRAIN_CSV, sep='\t')
valid_df = pd.read_csv(VALID_CSV, sep='\t')
test_df = pd.read_csv(TEST_CSV, sep='\t')

print(f"\nOriginal data:")
print(f"  Training: {len(train_df)} samples")
print(f"    Dementia: {len(train_df[train_df.label=='dementia'])} ({100*len(train_df[train_df.label=='dementia'])/len(train_df):.1f}%)")
print(f"    Nodementia: {len(train_df[train_df.label=='nodementia'])} ({100*len(train_df[train_df.label=='nodementia'])/len(train_df):.1f}%)")
print(f"  Validation: {len(valid_df)} samples")
print(f"  Test: {len(test_df)} samples")


LOADING DATA

Original data:
  Training: 480 samples
    Dementia: 185 (38.5%)
    Nodementia: 295 (61.5%)
  Validation: 69 samples
  Test: 72 samples


## OVERSAMPLE MINORITY CLASS

In [7]:
if USE_OVERSAMPLING:
    print("\n" + "="*70)
    print("OVERSAMPLING MINORITY CLASS")
    print("="*70)

    train_dementia = train_df[train_df['label'] == 'dementia']
    train_nodementia = train_df[train_df['label'] == 'nodementia']

    print(f"Before: Dementia={len(train_dementia)}, Nodementia={len(train_nodementia)}")

    # Oversample dementia to match nodementia
    train_dementia_oversampled = resample(
        train_dementia,
        replace=True,
        n_samples=len(train_nodementia),
        random_state=42
    )

    train_df = pd.concat([train_nodementia, train_dementia_oversampled])
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"After: Dementia={len(train_df[train_df.label=='dementia'])}, Nodementia={len(train_df[train_df.label=='nodementia'])}")
    print(f"Total training samples: {len(train_df)}")
    print(f"Balance: {100*len(train_df[train_df.label=='dementia'])/len(train_df):.1f}% dementia")
    print("✅ Training set balanced to 50/50!")


OVERSAMPLING MINORITY CLASS
Before: Dementia=185, Nodementia=295
After: Dementia=295, Nodementia=295
Total training samples: 590
Balance: 50.0% dementia
✅ Training set balanced to 50/50!


## LABEL MAPPING AND CLASS WEIGHTS

In [8]:
print("\n" + "="*70)
print("LABEL MAPPING AND CLASS WEIGHTS")
print("="*70)

label_list = sorted(train_df['label'].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print(f"Label mapping: {label2id}")

train_df['labels'] = train_df['label'].map(label2id)
valid_df['labels'] = valid_df['label'].map(label2id)
test_df['labels'] = test_df['label'].map(label2id)

# Compute class weights
if USE_CLASS_WEIGHTING:
    class_weights_array = compute_class_weight(
        'balanced',
        classes=np.unique(train_df['labels']),
        y=train_df['labels']
    )
    print(f"\nClass weights:")
    for i, weight in enumerate(class_weights_array):
        print(f"  {id2label[i]}: {weight:.3f}")
    print(f"Dementia misclassifications penalized {class_weights_array[0]/class_weights_array[1]:.2f}x more!")
else:
    class_weights_array = np.array([1.0, 1.0])
    print("Class weighting disabled")


LABEL MAPPING AND CLASS WEIGHTS
Label mapping: {'dementia': 0, 'nodementia': 1}

Class weights:
  dementia: 1.000
  nodementia: 1.000
Dementia misclassifications penalized 1.00x more!


## AUDIO AUGMENTATION

In [9]:
if USE_AUGMENTATION:
    print("\n" + "="*70)
    print("AUDIO AUGMENTATION")
    print("="*70)

    augment = A.Compose([
        A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.01, p=0.5),
        A.TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
        A.PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
        A.Shift(min_shift=-0.3, max_shift=0.3, p=0.5),
    ])

    print("Augmentation pipeline:")
    print("  - Gaussian noise (p=0.5)")
    print("  - Time stretch 0.8x-1.2x (p=0.5)")
    print("  - Pitch shift ±3 semitones (p=0.5)")
    print("  - Time shift ±30% (p=0.5)")
else:
    augment = None
    print("Augmentation disabled")

def load_and_preprocess_audio(path, target_sr=16000, max_duration=15.0, apply_augment=False):
    """Load and preprocess audio file."""
    waveform, sr = torchaudio.load(path)

    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)

    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    audio_np = waveform.squeeze().numpy()

    if apply_augment and augment is not None:
        audio_np = augment(samples=audio_np, sample_rate=target_sr)

    max_length = int(target_sr * max_duration)
    if len(audio_np) > max_length:
        audio_np = audio_np[:max_length]
    elif len(audio_np) < max_length:
        padding = max_length - len(audio_np)
        audio_np = np.pad(audio_np, (0, padding), mode='constant')

    return audio_np


AUDIO AUGMENTATION
Augmentation pipeline:
  - Gaussian noise (p=0.5)
  - Time stretch 0.8x-1.2x (p=0.5)
  - Pitch shift ±3 semitones (p=0.5)
  - Time shift ±30% (p=0.5)


## FEATURE EXTRACTION

In [10]:
print("\n" + "="*70)
print("LOADING FEATURE EXTRACTOR")
print("="*70)

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
feature_extractor.return_attention_mask = True

print(f"Feature extractor: {type(feature_extractor).__name__}")
print(f"Sampling rate: {feature_extractor.sampling_rate} Hz")

def preprocess_function(examples, apply_augment=False):
    """Preprocess batch of examples."""
    audio_arrays = []
    for path in examples["path"]:
        audio = load_and_preprocess_audio(
            path,
            target_sr=feature_extractor.sampling_rate,
            max_duration=MAX_DURATION,
            apply_augment=apply_augment and USE_AUGMENTATION
        )
        audio_arrays.append(audio)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        return_tensors="pt",
        padding=True,
        max_length=int(feature_extractor.sampling_rate * MAX_DURATION),
        truncation=True
    )

    return {k: v.numpy() for k, v in inputs.items()}


LOADING FEATURE EXTRACTOR


Feature extractor: Wav2Vec2FeatureExtractor
Sampling rate: 16000 Hz


## CREATE DATASETS

In [11]:
print("\n" + "="*70)
print("CREATING DATASETS")
print("="*70)

train_dataset = Dataset.from_pandas(train_df[['path', 'labels']])
valid_dataset = Dataset.from_pandas(valid_df[['path', 'labels']])
test_dataset = Dataset.from_pandas(test_df[['path', 'labels']])

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Valid dataset: {len(valid_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")


CREATING DATASETS
Train dataset: 590 samples
Valid dataset: 69 samples
Test dataset: 72 samples


## PREPROCESS DATASETS

In [12]:
print("\n" + "="*70)
print("PREPROCESSING DATASETS")
print("="*70)

print("Preprocessing training data (WITH augmentation)...")
encoded_train = train_dataset.map(
    lambda x: preprocess_function(x, apply_augment=True),
    batched=True,
    batch_size=8,
    remove_columns=["path"]
)

print("Preprocessing validation data (NO augmentation)...")
encoded_valid = valid_dataset.map(
    lambda x: preprocess_function(x, apply_augment=False),
    batched=True,
    batch_size=8,
    remove_columns=["path"]
)

print("Preprocessing test data (NO augmentation)...")
encoded_test = test_dataset.map(
    lambda x: preprocess_function(x, apply_augment=False),
    batched=True,
    batch_size=8,
    remove_columns=["path"]
)

print(f"✅ Preprocessing complete")
print(f"  Train: {len(encoded_train)} samples")
print(f"  Valid: {len(encoded_valid)} samples")
print(f"  Test: {len(encoded_test)} samples")


PREPROCESSING DATASETS
Preprocessing training data (WITH augmentation)...


Map: 100%|██████████| 590/590 [03:50<00:00,  2.55 examples/s]


Preprocessing validation data (NO augmentation)...


Map: 100%|██████████| 69/69 [00:13<00:00,  5.14 examples/s]


Preprocessing test data (NO augmentation)...


Map: 100%|██████████| 72/72 [00:16<00:00,  4.31 examples/s]

✅ Preprocessing complete
  Train: 590 samples
  Valid: 69 samples
  Test: 72 samples


## METRICS

In [13]:
def compute_metrics(eval_pred):
    """Compute comprehensive metrics."""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids

    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1]
    )

    # Macro averages
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro'
    )

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_dementia': f1_per_class[0],
        'f1_nodementia': f1_per_class[1],
        'recall_dementia': recall_per_class[0],
        'recall_nodementia': recall_per_class[1],
        'precision_dementia': precision_per_class[0],
        'precision_nodementia': precision_per_class[1],
    }

print("\n✅ Metrics function defined")
print("  Primary metric: F1-macro")
print("  Critical metric: Recall for dementia")


✅ Metrics function defined
  Primary metric: F1-macro
  Critical metric: Recall for dementia


## CUSTOM TRAINER WITH WEIGHTED LOSS

In [14]:
class WeightedTrainer(Trainer):
    """Trainer with class-weighted loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        if USE_CLASS_WEIGHTING:
            weight = torch.tensor(class_weights_array, device=logits.device, dtype=logits.dtype)
            loss_fn = nn.CrossEntropyLoss(weight=weight)
            loss = loss_fn(logits, labels)
        else:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

print("\n✅ Custom trainer defined")
if USE_CLASS_WEIGHTING:
    print(f"  Using weighted loss (weights: {class_weights_array})")
else:
    print("  Using standard loss")


✅ Custom trainer defined
  Using weighted loss (weights: [1. 1.])


## LOAD MODEL

In [15]:
print("\n" + "="*70)
print("LOADING MODEL")
print("="*70)

num_labels = len(label_list)
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

print(f"Model: {MODEL_CHECKPOINT}")
print(f"Labels: {num_labels}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


LOADING MODEL


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 1089.80it/s, Materializing param=wav2vec2.masked_spec_embed]                                            
Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because mis

Model: facebook/wav2vec2-base
Labels: 2
Parameters: 94,569,090
Trainable: 94,569,090


## TRAINING CONFIGURATION

In [16]:
print("\n" + "="*70)
print("TRAINING CONFIGURATION")
print("="*70)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    warmup_steps=100,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",  # PRIMARY METRIC
    push_to_hub=False,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_valid,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Best model metric: f1_macro")
print(f"Mixed precision (FP16): {training_args.fp16}")
print(f"Total training steps: {len(encoded_train) // BATCH_SIZE * EPOCHS}")


TRAINING CONFIGURATION
Output directory: ./results/wav2vec2-base-improved-v2
Epochs: 15
Batch size: 8
Learning rate: 2e-05
Best model metric: f1_macro
Mixed precision (FP16): True
Total training steps: 1095


## TRAINING

In [17]:
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Training for {EPOCHS} epochs")
print(f"Expected time: 30-60 minutes")
print("="*70)

try:
    train_result = trainer.train()

    print("\n" + "="*70)
    print("TRAINING COMPLETE!")
    print("="*70)
    print(f"Training loss: {train_result.metrics.get('train_loss', 'N/A')}")
    print(f"Training runtime: {train_result.metrics.get('train_runtime', 0)/60:.1f} minutes")

except Exception as e:
    print(f"\n❌ Training failed with error: {e}")
    raise


STARTING TRAINING
Training for 15 epochs
Expected time: 30-60 minutes


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Dementia,F1 Nodementia,Recall Dementia,Recall Nodementia,Precision Dementia,Precision Nodementia
1,0.686970,0.663289,0.753623,0.556522,0.260870,0.852174,0.176471,0.942308,0.500000,0.777778
2,0.639442,0.614298,0.695652,0.516517,0.222222,0.810811,0.176471,0.865385,0.300000,0.762712
3,0.534737,0.568990,0.753623,0.481203,0.105263,0.857143,0.058824,0.980769,0.500000,0.761194
4,0.526289,0.770195,0.420290,0.405172,0.310345,0.500000,0.529412,0.384615,0.219512,0.714286
5,0.525952,0.695062,0.594203,0.532882,0.363636,0.702128,0.470588,0.634615,0.296296,0.785714
6,0.413363,0.709740,0.710145,0.551948,0.285714,0.818182,0.235294,0.865385,0.363636,0.775862
7,0.141602,0.882440,0.710145,0.574074,0.333333,0.814815,0.294118,0.846154,0.384615,0.785714
8,0.201204,0.883347,0.724638,0.585258,0.344828,0.825688,0.294118,0.865385,0.416667,0.789474
9,0.159733,0.962219,0.753623,0.608609,0.370370,0.846847,0.294118,0.903846,0.500000,0.796610
10,0.046292,1.180759,0.724638,0.562563,0.296296,0.828829,0.235294,0.884615,0.400000,0.779661


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]



TRAINING COMPLETE!
Training loss: 0.2964793401728342
Training runtime: 43.1 minutes


## EVALUATION

In [18]:
print("\n" + "="*70)
print("EVALUATION")
print("="*70)

# Validation
print("\nEvaluating on validation set...")
valid_metrics = trainer.evaluate(encoded_valid)

print("\nVALIDATION METRICS:")
for key, value in sorted(valid_metrics.items()):
    if not key.startswith('eval_runtime') and not key.startswith('eval_samples'):
        print(f"  {key:25s}: {value:.4f}")

# Test
print("\nEvaluating on TEST set...")
test_metrics = trainer.evaluate(encoded_test)

print("\n" + "="*70)
print("🎯 FINAL TEST METRICS (UNBIASED)")
print("="*70)
for key, value in sorted(test_metrics.items()):
    if not key.startswith('eval_runtime') and not key.startswith('eval_samples'):
        print(f"  {key:25s}: {value:.4f}")


EVALUATION

Evaluating on validation set...



VALIDATION METRICS:
  epoch                    : 15.0000
  eval_accuracy            : 0.7536
  eval_f1_dementia         : 0.3704
  eval_f1_macro            : 0.6086
  eval_f1_nodementia       : 0.8468
  eval_loss                : 0.9622
  eval_precision_dementia  : 0.5000
  eval_precision_nodementia: 0.7966
  eval_recall_dementia     : 0.2941
  eval_recall_nodementia   : 0.9038
  eval_steps_per_second    : 0.5710

Evaluating on TEST set...

🎯 FINAL TEST METRICS (UNBIASED)
  epoch                    : 15.0000
  eval_accuracy            : 0.7639
  eval_f1_dementia         : 0.4516
  eval_f1_macro            : 0.6506
  eval_f1_nodementia       : 0.8496
  eval_loss                : 0.7972
  eval_precision_dementia  : 0.4667
  eval_precision_nodementia: 0.8421
  eval_recall_dementia     : 0.4375
  eval_recall_nodementia   : 0.8571
  eval_steps_per_second    : 0.4840


## CONFUSION MATRICES

In [19]:
print("\n" + "="*70)
print("GENERATING CONFUSION MATRICES")
print("="*70)

valid_predictions = trainer.predict(encoded_valid)
test_predictions = trainer.predict(encoded_test)

valid_pred_labels = np.argmax(valid_predictions.predictions, axis=1)
valid_true_labels = valid_predictions.label_ids

test_pred_labels = np.argmax(test_predictions.predictions, axis=1)
test_true_labels = test_predictions.label_ids

cm_valid = confusion_matrix(valid_true_labels, valid_pred_labels)
cm_test = confusion_matrix(test_true_labels, test_pred_labels)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_valid, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_list, yticklabels=label_list,
            cbar_kws={'label': 'Count'}, ax=axes[0])
axes[0].set_title(f'Validation Confusion Matrix\nAccuracy: {valid_metrics["eval_accuracy"]:.2%}',
                  fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_list, yticklabels=label_list,
            cbar_kws={'label': 'Count'}, ax=axes[1])
axes[1].set_title(f'Test Confusion Matrix (Final)\nAccuracy: {test_metrics["eval_accuracy"]:.2%}',
                  fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_v2.png'), dpi=300, bbox_inches='tight')
print(f"✅ Saved confusion matrices to {OUTPUT_DIR}/confusion_matrices_v2.png")

# Print classification reports
print("\nVALIDATION CLASSIFICATION REPORT:")
print(classification_report(valid_true_labels, valid_pred_labels, target_names=label_list, digits=4))

print("\nTEST CLASSIFICATION REPORT:")
print(classification_report(test_true_labels, test_pred_labels, target_names=label_list, digits=4))


GENERATING CONFUSION MATRICES


✅ Saved confusion matrices to ./results/wav2vec2-base-improved-v2/confusion_matrices_v2.png

VALIDATION CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    dementia     0.5000    0.2941    0.3704        17
  nodementia     0.7966    0.9038    0.8468        52

    accuracy                         0.7536        69
   macro avg     0.6483    0.5990    0.6086        69
weighted avg     0.7235    0.7536    0.7295        69


TEST CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    dementia     0.4667    0.4375    0.4516        16
  nodementia     0.8421    0.8571    0.8496        56

    accuracy                         0.7639        72
   macro avg     0.6544    0.6473    0.6506        72
weighted avg     0.7587    0.7639    0.7611        72



## SAVE RESULTS

In [20]:
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

# Save model
trainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# Save metrics
all_metrics = {
    "validation": valid_metrics,
    "test": test_metrics,
    "training": train_result.metrics
}

with open(os.path.join(OUTPUT_DIR, 'all_metrics_v2.json'), 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"✅ Metrics saved to {OUTPUT_DIR}/all_metrics_v2.json")

# Save configuration
config = {
    "model": MODEL_CHECKPOINT,
    "dataset": "combined" if USE_COMBINED_DATASET else "dementianet_only",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "max_duration": MAX_DURATION,
    "sampling_rate": SAMPLING_RATE,
    "use_class_weighting": USE_CLASS_WEIGHTING,
    "use_oversampling": USE_OVERSAMPLING,
    "use_augmentation": USE_AUGMENTATION,
    "class_weights": class_weights_array.tolist(),
    "train_samples": len(train_df),
    "valid_samples": len(valid_df),
    "test_samples": len(test_df),
    "label_mapping": label2id,
}

with open(os.path.join(OUTPUT_DIR, 'training_config_v2.json'), 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Config saved to {OUTPUT_DIR}/training_config_v2.json")


SAVING RESULTS


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.09it/s]

✅ Model saved to ./results/wav2vec2-base-improved-v2
✅ Metrics saved to ./results/wav2vec2-base-improved-v2/all_metrics_v2.json
✅ Config saved to ./results/wav2vec2-base-improved-v2/training_config_v2.json


## FINAL SUMMARY

In [21]:
print("\n" + "="*70)
print("FINAL SUMMARY - V2 IMPROVEMENTS")
print("="*70)

print(f"\nImprovements applied:")
print(f"  ✓ Class weighting: {USE_CLASS_WEIGHTING}")
if USE_CLASS_WEIGHTING:
    print(f"    Weights: dementia={class_weights_array[0]:.2f}, nodementia={class_weights_array[1]:.2f}")
print(f"  ✓ Oversampling: {USE_OVERSAMPLING}")
if USE_OVERSAMPLING:
    print(f"    Training balanced to 50/50")
print(f"  ✓ Augmentation: {USE_AUGMENTATION}")
print(f"  ✓ Longer audio: {MAX_DURATION}s (was 10s)")
print(f"  ✓ More epochs: {EPOCHS} (was 10)")
print(f"  ✓ F1-macro as primary metric (was accuracy)")

print(f"\nKey test metrics:")
print(f"  Accuracy: {test_metrics['eval_accuracy']:.4f}")
print(f"  F1-macro: {test_metrics['eval_f1_macro']:.4f}")
print(f"  Recall (dementia): {test_metrics['eval_recall_dementia']:.4f}  ← CRITICAL!")
print(f"  Recall (nodementia): {test_metrics['eval_recall_nodementia']:.4f}")

# Check if improvement over v1
if test_metrics['eval_recall_dementia'] > 0.1:
    print(f"\n✅ SUCCESS! Model now predicts dementia class")
    print(f"   (V1 had 0% recall, V2 has {100*test_metrics['eval_recall_dementia']:.1f}%)")
else:
    print(f"\n⚠️  Model still struggles with dementia class")
    print(f"   Consider: more data, different model architecture, or feature engineering")

print(f"\nFiles saved:")
print(f"  - {OUTPUT_DIR}/model.safetensors")
print(f"  - {OUTPUT_DIR}/all_metrics_v2.json")
print(f"  - {OUTPUT_DIR}/confusion_matrices_v2.png")
print(f"  - {OUTPUT_DIR}/training_config_v2.json")
print(f"  - {log_file}")

print(f"\n" + "="*70)
print(f"✅ TRAINING V2 COMPLETE!")
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"=" *70)

print(f"\n📊 Check results in: {OUTPUT_DIR}")
print(f"📝 Full log: {log_file}")


FINAL SUMMARY - V2 IMPROVEMENTS

Improvements applied:
  ✓ Class weighting: True
    Weights: dementia=1.00, nodementia=1.00
  ✓ Oversampling: True
    Training balanced to 50/50
  ✓ Augmentation: True
  ✓ Longer audio: 15.0s (was 10s)
  ✓ More epochs: 15 (was 10)
  ✓ F1-macro as primary metric (was accuracy)

Key test metrics:
  Accuracy: 0.7639
  F1-macro: 0.6506
  Recall (dementia): 0.4375  ← CRITICAL!
  Recall (nodementia): 0.8571

✅ SUCCESS! Model now predicts dementia class
   (V1 had 0% recall, V2 has 43.8%)

Files saved:
  - ./results/wav2vec2-base-improved-v2/model.safetensors
  - ./results/wav2vec2-base-improved-v2/all_metrics_v2.json
  - ./results/wav2vec2-base-improved-v2/confusion_matrices_v2.png
  - ./results/wav2vec2-base-improved-v2/training_config_v2.json


NameError: name 'log_file' is not defined